# ApexPINN clean M2 calibration pilot v1

Prepared, NOT RUN. Keep the existing F3 v1 training unchanged. Run this separate pilot after F3
has finished, on hosted Colab L4 with Python 3.12 and float64. This starts 25 new fits when launched:
20 tuning trials plus five final members, 1000 steps each. Duration and resource use are Unknown.

Post-hoc method declared 2026-11-06, prompted by v1 F1/F2 coverage differences. F3 v1 results were
not used to choose this method. Canonical declaration: configs/calibration_clean_v1.yaml.
For outer F3: tune on F4, reserve F5 for calibration, train tuning candidates on F1/F2/F6, then
fit the five final members on F1/F2/F4/F6. Never refit them after calibration. All four horizons
get separate event-balanced interval scales. No new fit is counted as a T13 v1 completion.

Raw/calibrated forecasts have identical means and targets and exact Gaussian-mixture variance.
Coverage, width and CRPS are reported per horizon and circuit. No coverage guarantee under circuit
shift or dependent laps is claimed. Comparing this pilot with v1 also changes its training data
and raw-noise estimator. No S1, v1 score, threshold, G2 or P2 decision is changed.


In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive, files
except ImportError as exc:
    raise RuntimeError('Use a hosted Colab GPU. Local execution is prohibited.') from exc
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('A hosted Colab runtime is required.')
drive.mount('/content/drive')
os.environ['APEXPINN_REMOTE_RUNTIME'] = 'colab'


## 2. Select a new, separate pilot output


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/apexpinn')
INPUT_BUNDLE = DRIVE_ROOT / 'inputs/apexpinn_t13_input_v1.zip'
BUNDLE_SHA256 = 'b787682f95e30b303bfe7a41a2739dd40051a200724f079ffcab84a7d5618a03'
OVERLAY_BUNDLE = DRIVE_ROOT / 'inputs/apexpinn_calibration_clean_v1.zip'
OVERLAY_SHA256 = 'd0495822b9d19abd5551c59380e1e9dc3aa817d1bd653e68ed7ec039db50e99f'
PILOT_ID = 'calib_clean_m2_f3_mid_v1'
RUN_ROOT = DRIVE_ROOT / 'calibration_runs' / PILOT_ID
EXPORT_TAG = 'checkpoint_01'
FOLD = 'F3'
PHI = 'mid'
F3_JOB = DRIVE_ROOT / 'runs/t13_m2_f3_mid_run01/jobs/M2_F3_mid_main/job.json'


## 3. Restore a separate verified workspace

Reuse the original input parts and a separately hashed code update. This workspace and its
checkpoints differ from the running F3 v1 workspace. No existing run directory is modified.


### Restore the input from Drive upload parts

The connector limits each file to 100 MiB. These two parts contain the unchanged original ZIP. This cell checks every part and the full archive SHA, then writes the ZIP on the remote VM. It performs file I/O only. No model is fitted. The original notebook and bundle are preserved.

In [ ]:
import hashlib
import json
import re
import tempfile

def assemble_input_bundle(input_bundle, expected_sha, destination_directory):
    manifest = json.loads(Path(str(input_bundle) + '.parts.json').read_text())
    if manifest['archive_sha256'] != expected_sha:
        raise ValueError('Part manifest differs from the approved input receipt.')
    checksum = hashlib.sha256()
    with tempfile.NamedTemporaryFile(
        mode='wb', prefix='apexpinn_input_', suffix='.zip',
        dir=destination_directory, delete=False
    ) as output:
        assembled = Path(output.name)
        for part in manifest['parts']:
            if not re.fullmatch(re.escape(input_bundle.name) + r'\.part[0-9]{2}', part['name']):
                raise ValueError('Unsafe input part name.')
            part_hash = hashlib.sha256()
            count = 0
            with (input_bundle.parent / part['name']).open('rb') as stream:
                for block in iter(lambda: stream.read(1 << 20), b''):
                    output.write(block)
                    checksum.update(block)
                    part_hash.update(block)
                    count += len(block)
            if count != part['bytes'] or part_hash.hexdigest() != part['sha256']:
                raise ValueError('Input part hash or size differs.')
    if checksum.hexdigest() != expected_sha:
        raise ValueError('Assembled archive differs from the approved input SHA-256.')
    return assembled

if not INPUT_BUNDLE.exists():
    INPUT_BUNDLE = assemble_input_bundle(INPUT_BUNDLE, BUNDLE_SHA256, Path('/content'))
    print('Reassembled and hash-verified input ZIP on the remote VM:', INPUT_BUNDLE)


In [ ]:
import hashlib
import json

def file_sha(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if file_sha(INPUT_BUNDLE) != BUNDLE_SHA256:
    raise ValueError('Input archive hash mismatch.')
sys.path.insert(0, str(INPUT_BUNDLE) + '/src')
from apexpinn.cloud.transfer import unpack
import importlib.util
from apexpinn.cloud.transfer import unpack

if file_sha(OVERLAY_BUNDLE) != OVERLAY_SHA256:
    raise ValueError('Update archive hash mismatch')
OVERLAY_ROOT = Path('/content') / ('apexpinn_update_' + OVERLAY_SHA256[:12])
if not OVERLAY_ROOT.exists():
    unpack(OVERLAY_BUNDLE, OVERLAY_ROOT, expected_sha256=OVERLAY_SHA256, kind='input')
else:
    import zipfile
    with zipfile.ZipFile(OVERLAY_BUNDLE) as update:
        expected_overlay = json.loads(update.read('bundle_manifest.json'))
    if json.loads((OVERLAY_ROOT / 'bundle_manifest.json').read_text()) != expected_overlay:
        raise ValueError('Update manifest changed')
    for name, item in expected_overlay['files'].items():
        if file_sha(OVERLAY_ROOT / name) != item['sha256']:
            raise ValueError('Update workspace changed')
spec = importlib.util.spec_from_file_location(
    'apexpinn_handoff', OVERLAY_ROOT / 'src/apexpinn/cloud/handoff.py')
handoff = importlib.util.module_from_spec(spec)
spec.loader.exec_module(handoff)
WORKSPACE = Path('/content') / ('apexpinn_' + BUNDLE_SHA256[:8] + '_' + OVERLAY_SHA256[:12])
handoff.restore_overlay(INPUT_BUNDLE, OVERLAY_ROOT, WORKSPACE, base_sha256=BUNDLE_SHA256)
print('Verified updated workspace:', WORKSPACE)


## 4. Install locked Python and verify L4/float64

This cell performs setup and device checks, not training. Actual device details and any error are
printed and saved to the persistent preflight log, before training can start.


In [ ]:
import shutil
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
UV = shutil.which('uv')
subprocess.run(
    [UV, 'sync', '--frozen', '--extra', 'cuda', '--no-dev', '--python', '3.12'],
    cwd=WORKSPACE, check=True)
PYTHON = WORKSPACE / '.venv/bin/python'

GPU_PROBE = "import json\nimport jax\njax.config.update('jax_enable_x64', True)\ndevices = [{'platform': d.platform, 'kind': d.device_kind} for d in jax.devices()]\nprint(json.dumps({'devices': devices, 'float64': jax.config.x64_enabled}), flush=True)\nfrom apexpinn.train.cloud_checkpoint import require_colab_gpu\nrequire_colab_gpu()\nif not all('L4' in d['kind'].upper() for d in devices):\n    raise RuntimeError('Select hosted L4. Detected: ' + str(devices))\n"
probe = subprocess.run(
    [str(PYTHON), '-m', 'apexpinn.cloud.process', '--cwd', str(WORKSPACE),
     '--logs', str(DRIVE_ROOT / 'logs' / PILOT_ID / 'preflight'), '--', str(PYTHON), '-c', GPU_PROBE],
    cwd=WORKSPACE, capture_output=True, text=True)
print(probe.stdout, end='')
print(probe.stderr, end='')
probe.check_returncode()


## 5. Review the clean split and new fit budget

This prints the plan only. Calibration and test circuits must be absent from tuning and fitting.
Twenty search trials and five final members are a new experiment, not retries of completed v1 slots.


In [ ]:
command = [str(PYTHON), 'scripts/run_calibration.py', '--out', str(RUN_ROOT),
           '--fold', FOLD, '--phi', PHI]
subprocess.run(command + ['--plan'], cwd=WORKSPACE, check=True)


## 6. Verify the existing trainer restart evidence

The unchanged numerical training engine already passed the v4 restart oracle. This checks those
engine hashes and does not run new optimizer updates. It does not claim the new orchestration
has been exercised end-to-end; the clean pilot itself remains NOT RUN until you launch it.


In [ ]:
oracle_path = DRIVE_ROOT / 'checks/t13_m1_f1_mid_patch01_0b6d9d66c3ba/selftest.json'
oracle = json.loads(oracle_path.read_text())
if not all(v.startswith('PASS:') for v in oracle['results'].values()):
    raise ValueError('Original trainer restart parity did not pass')
engine = {name: sha for name, sha in oracle['instrument']['source_sha256'].items()
          if name.startswith(('src/apexpinn/models/', 'src/apexpinn/physics/', 'src/apexpinn/data/'))
          or name in ('src/apexpinn/train/trainer.py', 'src/apexpinn/train/cloud_checkpoint.py',
                      'src/apexpinn/train/physics_study.py', 'src/apexpinn/eval/protocol.py',
                      'src/apexpinn/eval/prepared.py', 'src/apexpinn/eval/metrics.py')}
if not engine or any(file_sha(WORKSPACE / name) != sha for name, sha in engine.items()):
    raise ValueError('Numerical engine differs from the verified restart source')
print('Existing trainer restart evidence verified; no restart fits launched.')


## 7. Launch the clean pilot after F3 finishes

This cell starts real training on Colab. It refuses to launch before the completed F3 v1 job record
exists. Checkpoints and trial slots persist on Drive; rerun with compatible unchanged identity to
resume. No post-calibration refit is performed. Completed forecast bundles are reused without replay.


In [ ]:
if not F3_JOB.is_file():
    raise RuntimeError('Keep F3 v1 running. Wait for its completed job record before this pilot.')
finished = json.loads(F3_JOB.read_text())
if (finished.get('schema') != 'apexpinn.t13_job/v1'
        or finished['conditions']['model'] != 'M2' or finished['population']['fold'] != 'F3'
        or len(finished['final_fits']) != 5):
    raise ValueError('The expected completed F3 v1 job was not found')
subprocess.run([str(PYTHON), '-m', 'apexpinn.cloud.process', '--cwd', str(WORKSPACE),
                '--logs', str(DRIVE_ROOT / 'logs' / PILOT_ID / 'training'), '--', *command], check=True)


## 8. Review and export the separate result

The numerical evaluation is committed before rendering. If plotting failed after evaluation.json
was saved, this cell rebuilds the report without fitting or model inference. Original v1 results
stay unchanged. Compare interval coverage with width and CRPS, not pooled coverage alone.


In [ ]:
from IPython.display import Image, Markdown, display
if not (RUN_ROOT / 'evaluation.json').is_file():
    raise RuntimeError('No completed evaluation yet. Inspect the persistent process logs.')
if not (RUN_ROOT / 'report_receipt.json').is_file():
    subprocess.run(command + ['--report-only'], cwd=WORKSPACE, check=True)
display(Markdown((RUN_ROOT / 'calibration_report.md').read_text()))
display(Image(filename=str(RUN_ROOT / 'calibration_intervals.png')))
EXPORT = DRIVE_ROOT / 'exports' / (PILOT_ID + '_' + EXPORT_TAG + '.zip')
EXPORT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([str(PYTHON), '-m', 'apexpinn.cloud.transfer', 'pack-results',
                '--root', str(RUN_ROOT), '--out', str(EXPORT)], cwd=WORKSPACE, check=True)
print('Result archive:', EXPORT)
print('Receipt:', str(EXPORT) + '.receipt.json')
print('SHA-256:', file_sha(EXPORT))
